In [17]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb

# Processing Scraped Data - Grouping by Day

This notebook processes filtered scraped data to create 7-day activity vectors.

**Note:** We multiply the activity vectors by 1.2 to compensate for data gaps in the crawl.

In [18]:
# Initialize DuckDB connection
con = duckdb.connect()

# Load join dates (small table, can keep in memory)
user_of_interests = "../../data/ale_simplicistic_model/scraped/filtered/first_week_blockers.parquet"
con.execute("CREATE TABLE join_dates AS SELECT * FROM read_parquet(?)", [user_of_interests])

print("Join dates loaded")

Join dates loaded


In [19]:
# Helper: build 7-day vectors using DuckDB for efficient large-table processing
def make_vec_duckdb(file_path, left_on='did_id', vec_name='vec'):
    """
    Use DuckDB to:
    1. Read parquet file
    2. Join with join_dates
    3. Compute days_since_join
    4. Filter to first week (0-6 days)
    5. Aggregate counts per user per day
    6. Pivot to create 7-element vectors
    7. Multiply by 1.2 to compensate for data gaps
    """
    query = f"""
    WITH activity AS (
        SELECT 
            act.{left_on} as id_col,
            CAST(act.created_date AS TIMESTAMP) as created_date,
            CAST(jd.join_date AS TIMESTAMP) as join_date
        FROM read_parquet('{file_path}') act
        LEFT JOIN join_dates jd ON act.{left_on} = jd.did_id
    ),
    days_computed AS (
        SELECT 
            id_col,
            CAST(ROUND((EPOCH(created_date) - EPOCH(join_date)) / 86400.0) AS INTEGER) AS days_since_join
        FROM activity
        WHERE join_date IS NOT NULL
    ),
    first_week AS (
        SELECT id_col, days_since_join
        FROM days_computed
        WHERE days_since_join >= 0 AND days_since_join <= 6
    ),
    counts AS (
        SELECT 
            id_col,
            days_since_join,
            COUNT(*) as cnt
        FROM first_week
        GROUP BY id_col, days_since_join
    ),
    pivoted AS (
        SELECT
            id_col,
            MAX(CASE WHEN days_since_join = 0 THEN cnt ELSE 0 END) AS d0,
            MAX(CASE WHEN days_since_join = 1 THEN cnt ELSE 0 END) AS d1,
            MAX(CASE WHEN days_since_join = 2 THEN cnt ELSE 0 END) AS d2,
            MAX(CASE WHEN days_since_join = 3 THEN cnt ELSE 0 END) AS d3,
            MAX(CASE WHEN days_since_join = 4 THEN cnt ELSE 0 END) AS d4,
            MAX(CASE WHEN days_since_join = 5 THEN cnt ELSE 0 END) AS d5,
            MAX(CASE WHEN days_since_join = 6 THEN cnt ELSE 0 END) AS d6
        FROM counts
        GROUP BY id_col
    )
    SELECT 
        id_col AS did_id,
        [CAST(d0 * 1.2 AS INTEGER), 
         CAST(d1 * 1.2 AS INTEGER), 
         CAST(d2 * 1.2 AS INTEGER), 
         CAST(d3 * 1.2 AS INTEGER), 
         CAST(d4 * 1.2 AS INTEGER), 
         CAST(d5 * 1.2 AS INTEGER), 
         CAST(d6 * 1.2 AS INTEGER)] AS {vec_name}
    FROM pivoted
    """
    
    return con.execute(query).df()

print("DuckDB helper function defined (with 1.2x multiplier for data gaps)")

DuckDB helper function defined (with 1.2x multiplier for data gaps)


In [20]:
# Posts vectors using DuckDB
posts_path = "../../data/ale_simplicistic_model/scraped/filtered/posts.parquet"
posts_vec_df = make_vec_duckdb(posts_path, left_on='did_id', vec_name='posts_vec')

# Register the pandas DataFrame so DuckDB can reference it by name
con.register("posts_vec_df", posts_vec_df)

# Create uoi_activity with posts_vec already COALESCED to zeros when missing
con.execute("""
CREATE TABLE uoi_activity AS
SELECT jd.*,
       COALESCE(pv.posts_vec, [0,0,0,0,0,0,0]) AS posts_vec
FROM join_dates jd
LEFT JOIN posts_vec_df pv ON jd.did_id = pv.did_id
""")

print("Posts processed:")
print(con.execute("SELECT did_id, posts_vec FROM uoi_activity LIMIT 5").df())

Posts processed:
    did_id                  posts_vec
0   414918      [2, 1, 0, 5, 2, 0, 0]
1  2267190      [0, 1, 2, 0, 0, 0, 0]
2  2357567  [22, 14, 14, 14, 0, 7, 0]
3  2601555      [0, 1, 1, 0, 1, 0, 5]
4    95024      [0, 0, 0, 0, 0, 0, 1]


In [21]:
# Blocks: actor and subject 7-day vectors using DuckDB
blocks_path = "../../data/ale_simplicistic_model/scraped/filtered/blocks.parquet"

# Actor vectors
blocks_actor_vec = make_vec_duckdb(blocks_path, left_on='did_id', vec_name='blocks_actor_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(bav.blocks_actor_vec, [0,0,0,0,0,0,0]) AS blocks_actor_vec
    FROM uoi_activity ua
    LEFT JOIN blocks_actor_vec bav ON ua.did_id = bav.did_id
""")

# Subject vectors
blocks_subject_vec = make_vec_duckdb(blocks_path, left_on='subject_did_id', vec_name='blocks_subject_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(bsv.blocks_subject_vec, [0,0,0,0,0,0,0]) AS blocks_subject_vec
    FROM uoi_activity ua
    LEFT JOIN blocks_subject_vec bsv ON ua.did_id = bsv.did_id
""")

print("Blocks processed:")
print(con.execute("SELECT did_id, blocks_actor_vec, blocks_subject_vec FROM uoi_activity LIMIT 5").df())

Blocks processed:
    did_id       blocks_actor_vec     blocks_subject_vec
0  2438083  [0, 1, 1, 0, 0, 0, 1]  [1, 1, 2, 0, 0, 0, 0]
1  1151045  [1, 2, 2, 0, 0, 0, 0]  [0, 1, 1, 0, 0, 0, 0]
2  1771451  [1, 0, 1, 0, 0, 1, 0]  [2, 0, 0, 6, 0, 0, 0]
3  2482905  [0, 0, 0, 0, 1, 0, 0]  [0, 0, 0, 0, 1, 1, 0]
4  1135341  [1, 0, 0, 0, 0, 0, 0]  [0, 0, 1, 1, 0, 0, 0]


In [22]:
# Follows: actor and subject vectors using DuckDB
follows_path = "../../data/ale_simplicistic_model/scraped/filtered/follows.parquet"

# Actor vectors
follows_actor_vec = make_vec_duckdb(follows_path, left_on='did_id', vec_name='follows_actor_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(fav.follows_actor_vec, [0,0,0,0,0,0,0]) AS follows_actor_vec
    FROM uoi_activity ua
    LEFT JOIN follows_actor_vec fav ON ua.did_id = fav.did_id
""")

# Subject vectors
follows_subject_vec = make_vec_duckdb(follows_path, left_on='subject_did_id', vec_name='follows_subject_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(fsv.follows_subject_vec, [0,0,0,0,0,0,0]) AS follows_subject_vec
    FROM uoi_activity ua
    LEFT JOIN follows_subject_vec fsv ON ua.did_id = fsv.did_id
""")

print("Follows processed:")
print(con.execute("SELECT did_id, follows_actor_vec, follows_subject_vec FROM uoi_activity LIMIT 5").df())

Follows processed:
    did_id        follows_actor_vec    follows_subject_vec
0  2438083  [26, 1, 1, 0, 6, 1, 12]  [2, 0, 0, 0, 1, 0, 2]
1   185592    [0, 0, 2, 0, 1, 2, 1]  [0, 0, 0, 0, 0, 1, 1]
2  1707458    [0, 0, 0, 0, 0, 1, 0]  [1, 1, 1, 0, 1, 2, 0]
3   591427  [1, 0, 1, 0, 12, 0, 13]  [0, 0, 1, 0, 1, 0, 4]
4  2361852   [14, 0, 1, 0, 0, 1, 0]  [5, 1, 1, 0, 0, 0, 0]


In [23]:
# Likes: actor and subject vectors using DuckDB
likes_path = "../../data/ale_simplicistic_model/scraped/filtered/likes.parquet"

# Actor vectors
likes_actor_vec = make_vec_duckdb(likes_path, left_on='did_id', vec_name='likes_actor_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(lav.likes_actor_vec, [0,0,0,0,0,0,0]) AS likes_actor_vec
    FROM uoi_activity ua
    LEFT JOIN likes_actor_vec lav ON ua.did_id = lav.did_id
""")

# Subject vectors
likes_subject_vec = make_vec_duckdb(likes_path, left_on='subject_did_id', vec_name='likes_subject_vec')
con.execute("""
    CREATE OR REPLACE TABLE uoi_activity AS 
    SELECT 
        ua.*,
        COALESCE(lsv.likes_subject_vec, [0,0,0,0,0,0,0]) AS likes_subject_vec
    FROM uoi_activity ua
    LEFT JOIN likes_subject_vec lsv ON ua.did_id = lsv.did_id
""")

print("Likes processed:")
print(con.execute("SELECT did_id, likes_actor_vec, likes_subject_vec FROM uoi_activity LIMIT 5").df())

Likes processed:
    did_id                likes_actor_vec      likes_subject_vec
0   414918          [4, 5, 2, 0, 0, 1, 0]  [0, 0, 0, 0, 0, 1, 0]
1  2438083      [12, 6, 12, 1, 12, 1, 19]  [0, 0, 0, 0, 0, 0, 0]
2   185592          [1, 2, 4, 5, 0, 2, 1]  [0, 0, 0, 0, 0, 0, 0]
3   591427    [12, 13, 11, 16, 13, 8, 19]  [0, 0, 0, 0, 0, 0, 0]
4  2361852  [185, 88, 59, 59, 32, 83, 37]  [0, 0, 0, 0, 0, 0, 0]


## Save the file

In [24]:
# Export final result to parquet using DuckDB
save_path = "../../data/ale_simplicistic_model/scraped/processed/user_activity.parquet"

# Get final dataframe and save
final_df = con.execute("SELECT * FROM uoi_activity").df()

# Convert to pyarrow and save
import os
os.makedirs(os.path.dirname(save_path), exist_ok=True)

arrow_table = pa.Table.from_pandas(final_df, preserve_index=False)
pq.write_table(arrow_table, save_path, compression='zstd')

print('Saved user_table to', save_path)
print('Number of rows:', len(final_df))
print('Columns:', list(final_df.columns))
print('\nSample of final data:')
print(final_df.head())

# Close DuckDB connection
con.close()

Saved user_table to ../../data/ale_simplicistic_model/scraped/processed/user_activity.parquet
Number of rows: 1081
Columns: ['did_id', 'join_date', 'posts_vec', 'blocks_actor_vec', 'blocks_subject_vec', 'follows_actor_vec', 'follows_subject_vec', 'likes_actor_vec', 'likes_subject_vec']

Sample of final data:
    did_id   join_date                     posts_vec       blocks_actor_vec  \
0   414918  2026-01-08         [2, 1, 0, 5, 2, 0, 0]  [0, 1, 0, 0, 0, 0, 0]   
1  2438083  2026-01-02     [11, 6, 14, 7, 23, 1, 22]  [0, 1, 1, 0, 0, 0, 1]   
2   185592  2026-01-08         [4, 4, 4, 7, 1, 5, 4]  [0, 1, 0, 0, 0, 0, 0]   
3   591427  2026-01-04         [2, 0, 5, 2, 5, 5, 2]  [1, 1, 0, 0, 1, 0, 1]   
4  2361852  2026-01-04  [96, 17, 65, 38, 34, 44, 19]  [5, 0, 4, 1, 1, 0, 1]   

      blocks_subject_vec        follows_actor_vec    follows_subject_vec  \
0  [0, 0, 0, 0, 0, 0, 0]    [0, 0, 0, 0, 1, 0, 0]  [0, 0, 0, 1, 0, 2, 0]   
1  [1, 1, 2, 0, 0, 0, 0]  [26, 1, 1, 0, 6, 1, 12]  [2, 0, 0, 0,